<a href="https://colab.research.google.com/github/Ehsan-Roohi/DSMC_Python/blob/main/DSMC_Expansion_Wave.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# --- DSMC CODE - V16.1 SUB-CELL COLLISION (FIXED) ---
# این نسخه خطای تطابق نوع داده (AssertionError) در کتابخانه Numba را برطرف می‌کند.
# برخوردها در زیرسلول‌ها انجام می‌شود تا دقت فیزیکی افزایش یابد.

import numpy as np
import matplotlib.pyplot as plt
import numba
import time
from scipy.signal import savgol_filter
import pandas as pd

# ===================================================================
# ۱. بخش شبیه‌سازی (با تابع برخورد اصلاح‌شده)
# ===================================================================
MASS_AR = 39.948e-3 / 6.022e23; KB = 1.380649e-23

@numba.jit(nopython=True)
def calculate_vhs_cross_section_numba(vr_mag):
    # پارامترهای ثابت برای آرگون در این تابع تعریف شده‌اند تا نیاز به پاس دادن آن‌ها نباشد
    d_ref = 4.17e-10
    t_ref = 273.0
    omega_vhs = 0.81
    mass_ar = 39.948e-3 / 6.022e23
    kb = 1.380649e-23

    if vr_mag < 1e-9: return 1e-30
    exponent = omega_vhs - 0.5
    c_ref_sq = 2 * kb * t_ref / mass_ar
    gamma_val = 1.04533 # For omega = 0.81, Gamma(2.5 - 0.81) = Gamma(1.69)
    d_sq = (d_ref**2) * ((c_ref_sq / vr_mag**2)**exponent) * (1 / gamma_val)
    return np.pi * d_sq

# تابع برخورد که بر اساس زیرسلول کار می‌کند
@numba.jit(nopython=True)
def perform_collisions_in_cell_subcell(particles, indices_in_cell, cell_start_x, cell_width, num_sub_cells, cell_vol, dt, fnum, sigma_vr_max):
    num_particles_in_main_cell = len(indices_in_cell)
    if num_particles_in_main_cell < 2:
        return

    # 1. ذرات را در زیرسلول‌های خود دسته‌بندی کن
    sub_cell_width = cell_width / num_sub_cells

    # ✅ FIX: نوع داده به np.int64 تغییر کرد تا با نوع ایندکس‌های ورودی (که از argsort می‌آیند) مطابقت داشته باشد
    sub_cell_groups = [[np.int64(x) for x in range(0)] for _ in range(num_sub_cells)]

    for p_idx in indices_in_cell:
        relative_pos = particles[p_idx, 0] - cell_start_x
        sub_cell_idx = int(relative_pos / sub_cell_width)
        if 0 <= sub_cell_idx < num_sub_cells:
            sub_cell_groups[sub_cell_idx].append(p_idx)

    # 2. برخوردها را در هر زیرسلول به صورت جداگانه انجام بده
    sub_cell_vol = cell_vol / num_sub_cells
    if sub_cell_vol < 1e-30: return

    for sc_idx in range(num_sub_cells):
        indices_in_sub_cell = sub_cell_groups[sc_idx]
        num_particles_in_sub_cell = len(indices_in_sub_cell)

        if num_particles_in_sub_cell < 2:
            continue

        num_candidate_pairs = (num_particles_in_sub_cell * (num_particles_in_sub_cell - 1) * fnum * sigma_vr_max * dt) / (2.0 * sub_cell_vol)
        num_pairs_to_select = int(np.floor(num_candidate_pairs + np.random.rand()))

        for _ in range(num_pairs_to_select):
            idx1_local = np.random.randint(0, num_particles_in_sub_cell)
            p1_idx = indices_in_sub_cell[idx1_local]

            idx2_local = np.random.randint(0, num_particles_in_sub_cell)
            if idx1_local == idx2_local: continue
            p2_idx = indices_in_sub_cell[idx2_local]

            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            vr_mag = np.sqrt(vr[0]**2 + vr[1]**2 + vr[2]**2)
            if vr_mag < 1e-9: continue

            sigma_t = calculate_vhs_cross_section_numba(vr_mag)

            if np.random.rand() < (sigma_t * vr_mag) / sigma_vr_max:
                vcm = 0.5 * (particles[p1_idx, 1:4] + particles[p2_idx, 1:4])
                cos_chi = 2 * np.random.rand() - 1.0
                sin_chi = np.sqrt(1 - cos_chi**2)
                phi_chi = 2.0 * np.pi * np.random.rand()

                vr_prime = np.array([vr_mag * sin_chi * np.cos(phi_chi),
                                     vr_mag * sin_chi * np.sin(phi_chi),
                                     vr_mag * cos_chi])

                particles[p1_idx, 1:4] = vcm + 0.5 * vr_prime
                particles[p2_idx, 1:4] = vcm - 0.5 * vr_prime

def run_dsmc_simulation(sim_params):
    LX = sim_params['LX']; RHO_INIT = sim_params['RHO_INIT']; T_INIT = sim_params['T_INIT']
    NUM_CELLS_X = sim_params['NUM_CELLS_X']; PARTICLES_PER_CELL_INIT = sim_params['PARTICLES_PER_CELL_INIT']
    TOTAL_TIME = sim_params['TOTAL_TIME']; DT = sim_params['DT']
    MIN_PARTICLES_FOR_STATS = 20
    NUM_SUB_CELLS_PER_CELL = sim_params['NUM_SUB_CELLS_PER_CELL']

    TOTAL_PARTICLES_SIM = (NUM_CELLS_X // 2) * PARTICLES_PER_CELL_INIT
    N_DENSITY_REAL = RHO_INIT / MASS_AR
    # فرض یک حجم سه بعدی برای محاسبه FNUM
    CELL_VOLUME_CONCEPTUAL = (LX / NUM_CELLS_X) * ((LX/10) * (LX/10))
    FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT
    NUM_STEPS = int(TOTAL_TIME / DT); SAMPLING_INTERVAL = int(NUM_STEPS / 4)

    particles = initialize_gas_expansion(TOTAL_PARTICLES_SIM, NUM_CELLS_X, LX, PARTICLES_PER_CELL_INIT, T_INIT)

    results_history = {}
    cell_width = LX / NUM_CELLS_X

    results_history[0.0] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

    vr_max_estimate = 5.0 * np.sqrt(KB * T_INIT / MASS_AR)
    sigma_at_vr_max = calculate_vhs_cross_section_numba(vr_max_estimate)
    SIGMA_VR_MAX_GLOBAL = sigma_at_vr_max * vr_max_estimate

    for step in range(1, NUM_STEPS + 1):
        particles[:, 0] += particles[:, 1] * DT

        hit_left = particles[:, 0] < 0; particles[hit_left, 1] *= -1; particles[hit_left, 0] *= -1
        hit_right = particles[:, 0] > LX; particles[hit_right, 1] *= -1; particles[hit_right, 0] = 2 * LX - particles[hit_right, 0]

        cell_indices = (particles[:, 0] / cell_width).astype(np.int64)
        cell_indices = np.clip(cell_indices, 0, NUM_CELLS_X - 1)
        sorted_particle_indices = np.argsort(cell_indices)
        cell_counts = np.bincount(cell_indices, minlength=NUM_CELLS_X)
        cell_start_indices = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(cell_counts[:-1])))

        for i in range(NUM_CELLS_X):
            start = cell_start_indices[i]; end = start + cell_counts[i]
            indices_in_cell_i = sorted_particle_indices[start:end]
            cell_start_x = i * cell_width

            perform_collisions_in_cell_subcell(particles, indices_in_cell_i, cell_start_x, cell_width, NUM_SUB_CELLS_PER_CELL, CELL_VOLUME_CONCEPTUAL, DT, FNUM, SIGMA_VR_MAX_GLOBAL)

        if step % SAMPLING_INTERVAL == 0 or step == NUM_STEPS:
            results_history[step * DT] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

    return results_history

def initialize_gas_expansion(total_particles, num_cells, lx, ppc, t_init):
    particles = np.zeros((total_particles, 4))
    cell_width = lx / num_cells
    num_occupied_cells = num_cells // 2
    for i in range(num_occupied_cells):
        start_idx = i * ppc; end_idx = (i + 1) * ppc
        particles[start_idx:end_idx, 0] = i * cell_width + np.random.rand(ppc) * cell_width
    v_thermal_std = np.sqrt(KB * t_init / MASS_AR)
    particles[:, 1:4] = np.random.normal(0, v_thermal_std, (total_particles, 3))
    particles[:, 1:4] -= np.mean(particles[:, 1:4], axis=0)
    return particles

def sample_properties(particles_state, num_cells, cell_width, fnum, cell_vol, min_parts):
    density_profile = np.zeros(num_cells)
    velocity_profile = np.full(num_cells, np.nan)
    temp_profile = np.full(num_cells, np.nan)

    cell_indices = (particles_state[:, 0] / cell_width).astype(np.int64)
    cell_indices = np.clip(cell_indices, 0, num_cells - 1)

    # برای بهینه‌سازی، از bincount برای شمارش و argsort برای دسته‌بندی استفاده می‌کنیم
    sorted_indices = np.argsort(cell_indices)
    counts = np.bincount(cell_indices, minlength=num_cells)
    starts = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(counts[:-1])))

    for i in range(num_cells):
        num_in_cell = counts[i]
        if num_in_cell > 0:
            density_profile[i] = num_in_cell * fnum / cell_vol

        if num_in_cell >= min_parts:
            start_pos = starts[i]
            end_pos = start_pos + num_in_cell
            indices_in_cell_i = sorted_indices[start_pos:end_pos]

            cell_velocities = particles_state[indices_in_cell_i, 1:4]
            mean_vel_cell = np.mean(cell_velocities, axis=0)
            velocity_profile[i] = mean_vel_cell[0]

            thermal_vel_sq = np.sum((cell_velocities - mean_vel_cell)**2)
            temp_profile[i] = (MASS_AR * thermal_vel_sq) / (3 * KB * num_in_cell) if num_in_cell > 1 else 0.0

    return {'density': density_profile, 'velocity': velocity_profile, 'temperature': temp_profile}

# ===================================================================
# ۲. بخش اصلی اجرا کننده و رسم نمودار
# ===================================================================
if __name__ == "__main__":
    SIMULATION_PARAMS = {
        'LX': 1.0e-6, 'RHO_INIT': 1.78, 'T_INIT': 273.0,
        'NUM_CELLS_X': 200,
        'PARTICLES_PER_CELL_INIT': 10000,
        'TOTAL_TIME': 0.4e-9, 'DT': 5.0e-12,
        'NUM_SUB_CELLS_PER_CELL': 6
    }
    NUM_ENSEMBLE_RUNS = 50

    print(f"--- شروع اجرای نهایی با مدل برخورد زیرسلولی (تعداد اجرا: {NUM_ENSEMBLE_RUNS}) ---")

    all_results = []
    start_time_total = time.time()
    for i in range(NUM_ENSEMBLE_RUNS):
        np.random.seed(int(time.time()) + i)
        single_run_history = run_dsmc_simulation(SIMULATION_PARAMS)
        all_results.append(single_run_history)
        print(f"--- اجرای {i+1}/{NUM_ENSEMBLE_RUNS} تمام شد ---")

    averaged_results = {}
    sample_times = sorted(all_results[0].keys())
    for t in sample_times:
        all_densities = np.array([run[t]['density'] for run in all_results if t in run])
        all_velocities = np.array([run[t]['velocity'] for run in all_results if t in run])
        all_temperatures = np.array([run[t]['temperature'] for run in all_results if t in run])
        averaged_results[t] = {
            'density': np.nanmean(all_densities, axis=0),
            'velocity': np.nanmean(all_velocities, axis=0),
            'temperature': np.nanmean(all_temperatures, axis=0),
        }

    cell_width = SIMULATION_PARAMS['LX'] / SIMULATION_PARAMS['NUM_CELLS_X']
    cell_centers = (np.arange(SIMULATION_PARAMS['NUM_CELLS_X']) + 0.5) * cell_width

    fig, axes = plt.subplots(3, 1, figsize=(12, 18), sharex=True)
    fig.suptitle(f"DSMC with Sub-cell Collisions (N_runs={NUM_ENSEMBLE_RUNS}, N_p_cell={SIMULATION_PARAMS['PARTICLES_PER_CELL_INIT']})", fontsize=16)

    for t in sample_times:
        data = averaged_results[t]
        label_text = f't = {t*1e9:.2f} ns'
        line_style = '--' if t == 0.0 else '-'

        s_dens = pd.Series(data['density']).interpolate(method='linear')
        axes[0].plot(cell_centers, s_dens, linestyle=line_style, label=label_text)

        s_vel = pd.Series(data['velocity']).interpolate(method='linear').fillna(0)
        vel_smoothed = savgol_filter(s_vel.to_numpy(), window_length=21, polyorder=2)
        axes[1].plot(cell_centers, vel_smoothed, linestyle=line_style, label=label_text)

        s_temp = pd.Series(data['temperature']).interpolate(method='linear').bfill().ffill()
        temp_smoothed = savgol_filter(s_temp.to_numpy(), window_length=21, polyorder=2)
        axes[2].plot(cell_centers, temp_smoothed, linestyle=line_style, label=label_text)

    axes[0].set_ylabel('Number Density ($m^{-3}$)'); axes[0].set_title('Density Profile'); axes[0].grid(True, linestyle=':'); axes[0].legend()
    axes[1].set_ylabel('Bulk Velocity (m/s)'); axes[1].set_title('Bulk Velocity Profile (Smoothed)'); axes[1].grid(True, linestyle=':'); axes[1].legend()
    axes[2].set_ylabel('Temperature (K)'); axes[2].set_title('Temperature Profile (Smoothed - Sub-cell Collisions)'); axes[2].grid(True, linestyle=':'); axes[2].legend()
    axes[2].set_xlabel('Position x (m)')

    axes[1].set_ylim(bottom=-50)
    axes[2].set_ylim(bottom=150, top=SIMULATION_PARAMS['T_INIT'] + 20)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("dsmc_v16_1_subcell_result.png", dpi=300)
    plt.show()